# Dhara - mT5-base provision-summarization fine-tuning (SSH)

Fine-tune a retrieve-then-summarize model: a provision goes in, a plain-Bangla
2-4 sentence summary comes out, grounded to that one provision only. This is
**not** a RAG wrapper around a frozen checkpoint - the weights are trained
here (LoRA, T5 seq2seq) - see DECISIONS.md's "not a RAG wrapper" entry for
why that distinction is load-bearing against the project's earlier
no-generation-layer ruling.

Primary checkpoint: `google/mt5-base`. Comparison checkpoint: `csebuetnlp/banglat5`
(Bangla-only, no cross-lingual capacity - useful contrast since a real slice
of provisions in this corpus are English-source Acts). Toggle `CHECKPOINT`
below to switch; every other cell is checkpoint-agnostic.

> Do not upload the raw provision text or this notebook's outputs to a public
> repository. Provision text itself is public law, but keep the review-status
> fields (`needs_human_check`) attached wherever this data travels.

## Required private input

`data/processed/summaries_train_v1.jsonl`, `summaries_val_v1.jsonl`,
`summaries_test_v1.jsonl` - built by `scripts/70_build_provision_summaries.py`
+ `scripts/71_split_summaries.py`, split by whole Act (no row-level leakage).
Every row has `source_text`, `summary_bn`, `domain`, `source_language`,
`act_id`, plus `label_source: "llm_authored_v1"` and
`review_status: "needs_human_check"` - these are Claude-authored candidate
summaries against real provision text, not yet human-verified. Train on them
honestly labelled as such; do not report them as the spec's "3,000
human-verified" target until an annotator has actually reviewed them.

In [ ]:
# Local runtime setup. Select the kernel from envs/ml before running this notebook.
import sys, importlib.util
required = ['transformers', 'peft', 'accelerate', 'datasets', 'torch', 'rouge_score', 'sacrebleu']
missing = [name for name in required if importlib.util.find_spec(name) is None]
assert not missing, 'Missing packages in envs/ml: ' + ', '.join(missing)
import torch
assert torch.cuda.is_available(), 'This notebook requires a CUDA GPU.'
print('Using', sys.executable, '| GPU:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import os
WORKDIR = Path(os.environ.get('DHARA_WORKDIR', Path.cwd())).expanduser().resolve()
DATA = Path(os.environ.get('DHARA_DATA', WORKDIR / 'data' / 'processed')).expanduser().resolve()
if not (DATA / 'summaries_train_v1.jsonl').exists() and (WORKDIR / 'summaries_train_v1.jsonl').exists():
    DATA = WORKDIR
print({'workdir': str(WORKDIR), 'data': str(DATA)})

In [ ]:
import json
def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train = read_jsonl(DATA / 'summaries_train_v1.jsonl')
val = read_jsonl(DATA / 'summaries_val_v1.jsonl')
test = read_jsonl(DATA / 'summaries_test_v1.jsonl')
for name, rows in [('train', train), ('val', val), ('test', test)]:
    assert rows, name + ' is empty'
    for r in rows:
        assert r['source_text'].strip() and r['summary_bn'].strip()

train_acts = {r['act_id'] for r in train}
val_acts = {r['act_id'] for r in val}
test_acts = {r['act_id'] for r in test}
assert not (train_acts & val_acts), 'Act leaked between train/val'
assert not (train_acts & test_acts), 'Act leaked between train/test'
assert not (val_acts & test_acts), 'Act leaked between val/test'
print({'train': len(train), 'val': len(val), 'test': len(test),
       'train_acts': len(train_acts), 'val_acts': len(val_acts), 'test_acts': len(test_acts)})

## Data gate - read before training

213 total rows (168/26/19) against the spec's minimum of 3,000
human-verified pairs (300-500+ per major domain). This is not enough to
expect real cross-domain generalization - the project's own spec says so
plainly: "Below roughly 1,000 reviewed summaries, the model may learn
formatting but will not reliably generalize across legal domains and
provision types." Running this notebook now is legitimate for two honest
reasons and no others: (1) validate the training infrastructure end-to-end
before spending more authoring effort, (2) get a real, reportable number
instead of a guess for where the R&D loop currently stands. Do not report
this run's numbers as the dataset's final result, and do not stop growing
`summaries_v1` because this run "worked".

In [ ]:
# Checkpoint toggle. Re-run from here down after switching.
CHECKPOINT = 'google/mt5-base'   # comparison run: 'csebuetnlp/banglat5'
RUN_NAME = CHECKPOINT.split('/')[-1].replace('-', '_')
SRC_LEN = 896     # spec: 768-1024
TGT_LEN = 192     # spec: 128-256
print({'checkpoint': CHECKPOINT, 'run_name': RUN_NAME})

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def make_prompt(row):
    return ('summarize in plain Bangla: ' + (row['provision_title_bn'] or '') + ' ' + row['source_text']).strip()

def tokenize_batch(rows):
    inputs = tokenizer([make_prompt(r) for r in rows], max_length=SRC_LEN, truncation=True,
                        padding='max_length', return_tensors='pt')
    with tokenizer.as_target_tokenizer():
        labels = tokenizer([r['summary_bn'] for r in rows], max_length=TGT_LEN, truncation=True,
                            padding='max_length', return_tensors='pt')
    label_ids = labels['input_ids']
    label_ids[label_ids == tokenizer.pad_token_id] = -100
    return {'input_ids': inputs['input_ids'], 'attention_mask': inputs['attention_mask'], 'labels': label_ids}

In [ ]:
import torch
from rouge_score import rouge_scorer
import sacrebleu

class WhitespaceTokenizer:
    def tokenize(self, text): return text.split()

_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False, tokenizer=WhitespaceTokenizer())

@torch.no_grad()
def generate_summaries(model, rows, batch_size=8, max_new_tokens=TGT_LEN):
    model.eval()
    preds = []
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i + batch_size]
        enc = tokenizer([make_prompt(r) for r in batch], max_length=SRC_LEN, truncation=True,
                         padding=True, return_tensors='pt').to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, num_beams=4)
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return preds

def evaluate(model, rows, run_id):
    preds = generate_summaries(model, rows)
    refs = [r['summary_bn'] for r in rows]
    rouge = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    per_query = []
    for row, pred, ref in zip(rows, preds, refs):
        result = _scorer.score(ref, pred)
        for key in rouge:
            rouge[key].append(result[key].fmeasure)
        per_query.append({'id': row['id'], 'provision_id': row['provision_id'], 'domain': row['domain'],
                           'reference': ref, 'prediction': pred})
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    import numpy as np
    return {
        'run_id': run_id, 'n': len(rows),
        'rouge1_f': float(np.mean(rouge['rouge1'])), 'rouge2_f': float(np.mean(rouge['rouge2'])),
        'rougeL_f': float(np.mean(rouge['rougeL'])), 'chrf': float(chrf), 'per_query': per_query,
    }

## Zero-shot baseline

Record this before any fine-tuning, same rule as the retrieval notebook's
zero-shot BGE-m3 control - it is what separates "the pretrained model can
already do this" from "the fine-tune taught it something." 

In [ ]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(CHECKPOINT).to('cuda')
zeroshot_result = evaluate(base_model, test, RUN_NAME + '_zeroshot')
print({k: v for k, v in zeroshot_result.items() if k != 'per_query'})
del base_model
torch.cuda.empty_cache()

In [ ]:
# LoRA fine-tuning. Spec: LoRA lr 1e-4, warmup 5-10%, label smoothing 0.1,
# early stopping on val loss, mixed precision, gradient accumulation.
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

SEED = 42
torch.manual_seed(SEED)

model = AutoModelForSeq2SeqLM.from_pretrained(CHECKPOINT).to('cuda')
lora = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=16, lora_alpha=32, lora_dropout=0.05,
                   bias='none', target_modules=['q', 'v'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

def to_dataset(rows):
    tok = tokenize_batch(rows)
    return Dataset.from_dict({k: v.tolist() for k, v in tok.items()})

train_ds = to_dataset(train)
val_ds = to_dataset(val)

args = Seq2SeqTrainingArguments(
    output_dir=str(WORKDIR / 'outputs' / (RUN_NAME + '_lora')),
    num_train_epochs=5, per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=1e-4, warmup_ratio=0.08, label_smoothing_factor=0.1, fp16=True,
    max_grad_norm=1.0, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='eval_loss',
    logging_steps=10, report_to='none', seed=SEED,
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorForSeq2Seq(tokenizer, model=model))
trainer.train()

In [ ]:
# Merge LoRA, evaluate once on frozen test, save results/runs json with per_query
# (CLAUDE.md: no hand-typed numbers, every number from a results/runs/*.json).
merged = model.merge_and_unload()
finetuned_result = evaluate(merged, test, RUN_NAME + '_finetuned_v1')
print({k: v for k, v in finetuned_result.items() if k != 'per_query'})
print({k: v for k, v in zeroshot_result.items() if k != 'per_query'})

import json as _json
OUT_DIR = WORKDIR / 'results' / 'runs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / (RUN_NAME + '_zeroshot.json')).write_text(_json.dumps(zeroshot_result, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT_DIR / (RUN_NAME + '_finetuned_v1.json')).write_text(_json.dumps(finetuned_result, ensure_ascii=False, indent=2), encoding='utf-8')

merged.save_pretrained(str(WORKDIR / 'outputs' / (RUN_NAME + '_finetuned_v1_merged')))
tokenizer.save_pretrained(str(WORKDIR / 'outputs' / (RUN_NAME + '_finetuned_v1_merged')))
print('saved to', WORKDIR / 'outputs' / (RUN_NAME + '_finetuned_v1_merged'))

## If ROUGE/chrF barely move vs. zero-shot

Same diagnosis order as the retrieval notebook's "still below target" section:
this is a 213-row dataset against a 3,000-row spec, so a null or weak result
here first tests whether the loop runs, not whether the model can learn the
task. Before concluding anything about the approach: (1) copy the extractive
baselines from `results/runs/summary_baselines_v1.json` into the same
comparison table so this number has a floor to sit above, (2) run the
`csebuetnlp/banglat5` comparison with the same cells (only `CHECKPOINT`
changes), (3) grow `summaries_v1` toward 1,000+ rows before trusting any
generalization claim, (4) get a human reviewer to flip `review_status` on a
sample so a factuality number (hallucination rate, number/date preservation)
means something beyond the `numbers_preserved` heuristic already in the
schema.